In [1]:
"""
04_centralized_baseline.py
Simple POI Recommendation System (Centralized - Non-Federated)
"""

import pandas as pd
import numpy as np
import pickle
from sklearn.decomposition import NMF
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt
import os

print("="*60)
print("CENTRALIZED POI RECOMMENDATION SYSTEM")
print("="*60)

# Navigate to project root
os.chdir('/Users/srivanur/Documents/fedpoi_thesis')

# ==========================================
# STEP 1: LOAD PREPROCESSED DATA
# ==========================================
print("\nStep 1: Loading preprocessed data...")

df = pd.read_csv('data/processed/foursquare_filtered.csv')
interaction_matrix = pd.read_csv('data/processed/interaction_matrix.csv', index_col=0)

print(f"✓ Loaded {len(df)} check-ins")
print(f"✓ Interaction matrix shape: {interaction_matrix.shape}")
print(f"  - Users: {interaction_matrix.shape[0]}")
print(f"  - Venues: {interaction_matrix.shape[1]}")

# Calculate sparsity
total_cells = interaction_matrix.shape[0] * interaction_matrix.shape[1]
non_zero_cells = interaction_matrix.sum().sum()
sparsity = 100 * (1 - non_zero_cells / total_cells)
print(f"  - Sparsity: {sparsity:.2f}%")

# ==========================================
# STEP 2: SPLIT DATA (Train/Test)
# ==========================================
print("\nStep 2: Splitting data into train/test...")

# Convert to numpy array
matrix_array = interaction_matrix.values

# For each user, hold out 20% of their visited venues for testing
train_matrix = np.zeros_like(matrix_array)
test_matrix = np.zeros_like(matrix_array)

for user_idx in range(matrix_array.shape[0]):
    # Get venues this user visited
    visited_venues = np.where(matrix_array[user_idx] == 1)[0]
    
    if len(visited_venues) >= 2:  # Need at least 2 visits to split
        # Split 80% train, 20% test
        n_test = max(1, int(len(visited_venues) * 0.2))
        test_venues = np.random.choice(visited_venues, size=n_test, replace=False)
        train_venues = np.setdiff1d(visited_venues, test_venues)
        
        # Fill matrices
        train_matrix[user_idx, train_venues] = 1
        test_matrix[user_idx, test_venues] = 1
    else:
        # If user has only 1 visit, put it in train
        train_matrix[user_idx] = matrix_array[user_idx]

print(f"✓ Train matrix: {train_matrix.sum()} interactions")
print(f"✓ Test matrix: {test_matrix.sum()} interactions")

# ==========================================
# STEP 3: BUILD RECOMMENDATION MODEL
# ==========================================
print("\nStep 3: Training recommendation model...")
print("Using Matrix Factorization (NMF)...")

# Matrix Factorization using NMF (Non-negative Matrix Factorization)
n_components = 20  # Latent factors (embedding size)

model = NMF(
    n_components=n_components,
    init='random',
    random_state=42,
    max_iter=200,
    verbose=0
)

# Train on training matrix
user_features = model.fit_transform(train_matrix)
venue_features = model.components_

print(f"✓ Model trained!")
print(f"  - User features shape: {user_features.shape}")
print(f"  - Venue features shape: {venue_features.shape}")

# ==========================================
# STEP 4: MAKE PREDICTIONS
# ==========================================
print("\nStep 4: Making predictions...")

# Reconstruct full matrix (predicted ratings)
predicted_matrix = np.dot(user_features, venue_features)

print(f"✓ Predicted matrix shape: {predicted_matrix.shape}")

# ==========================================
# STEP 5: EVALUATE PERFORMANCE
# ==========================================
print("\nStep 5: Evaluating model performance...")

def precision_at_k(predicted, actual, k=5):
    """
    Calculate Precision@K
    For each user, recommend top K venues and see how many were actually visited
    """
    precisions = []
    
    for user_idx in range(predicted.shape[0]):
        # Get actual venues user visited in test set
        actual_venues = set(np.where(actual[user_idx] == 1)[0])
        
        if len(actual_venues) == 0:  # Skip users with no test data
            continue
        
        # Get top K predicted venues (excluding already visited in train)
        train_venues = set(np.where(train_matrix[user_idx] == 1)[0])
        scores = predicted[user_idx].copy()
        scores[list(train_venues)] = -np.inf  # Don't recommend already visited
        
        top_k = np.argsort(scores)[-k:][::-1]
        
        # Calculate precision
        hits = len(set(top_k) & actual_venues)
        precision = hits / k
        precisions.append(precision)
    
    return np.mean(precisions)

def recall_at_k(predicted, actual, k=5):
    """
    Calculate Recall@K
    Of all venues user actually visited, how many did we recommend?
    """
    recalls = []
    
    for user_idx in range(predicted.shape[0]):
        actual_venues = set(np.where(actual[user_idx] == 1)[0])
        
        if len(actual_venues) == 0:
            continue
        
        train_venues = set(np.where(train_matrix[user_idx] == 1)[0])
        scores = predicted[user_idx].copy()
        scores[list(train_venues)] = -np.inf
        
        top_k = np.argsort(scores)[-k:][::-1]
        
        # Calculate recall
        hits = len(set(top_k) & actual_venues)
        recall = hits / len(actual_venues)
        recalls.append(recall)
    
    return np.mean(recalls)

# Calculate metrics for different K values
k_values = [5, 10, 20]
results = {}

print("\nEvaluation Metrics:")
print("-" * 40)

for k in k_values:
    precision = precision_at_k(predicted_matrix, test_matrix, k=k)
    recall = recall_at_k(predicted_matrix, test_matrix, k=k)
    
    results[k] = {'precision': precision, 'recall': recall}
    
    print(f"K = {k:2d}")
    print(f"  Precision@{k}: {precision:.4f} ({precision*100:.2f}%)")
    print(f"  Recall@{k}:    {recall:.4f} ({recall*100:.2f}%)")
    print()

# ==========================================
# STEP 6: SAVE MODEL AND RESULTS
# ==========================================
print("Step 6: Saving model and results...")

# Create results directory
os.makedirs('results/baseline', exist_ok=True)

# Save model components
np.save('results/baseline/user_features.npy', user_features)
np.save('results/baseline/venue_features.npy', venue_features)

# Save results
with open('results/baseline/metrics.pkl', 'wb') as f:
    pickle.dump(results, f)

# Save summary
with open('results/baseline/summary.txt', 'w') as f:
    f.write("CENTRALIZED BASELINE RESULTS\n")
    f.write("="*50 + "\n\n")
    f.write(f"Dataset: Foursquare NYC\n")
    f.write(f"Users: {train_matrix.shape[0]}\n")
    f.write(f"Venues: {train_matrix.shape[1]}\n")
    f.write(f"Train interactions: {train_matrix.sum()}\n")
    f.write(f"Test interactions: {test_matrix.sum()}\n")
    f.write(f"Sparsity: {sparsity:.2f}%\n")
    f.write(f"Model: Matrix Factorization (NMF)\n")
    f.write(f"Latent factors: {n_components}\n\n")
    f.write("RESULTS:\n")
    f.write("-"*50 + "\n")
    for k in k_values:
        f.write(f"K={k}: Precision={results[k]['precision']:.4f}, Recall={results[k]['recall']:.4f}\n")

print("✓ Saved to results/baseline/")

# ==========================================
# STEP 7: VISUALIZE RESULTS
# ==========================================
print("\nStep 7: Creating visualizations...")

# Plot 1: Precision and Recall vs K
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

k_list = list(results.keys())
precision_list = [results[k]['precision'] for k in k_list]
recall_list = [results[k]['recall'] for k in k_list]

ax1.plot(k_list, precision_list, marker='o', linewidth=2, markersize=8, color='blue')
ax1.set_xlabel('K (Number of Recommendations)', fontsize=12)
ax1.set_ylabel('Precision@K', fontsize=12)
ax1.set_title('Precision vs K', fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

ax2.plot(k_list, recall_list, marker='s', linewidth=2, markersize=8, color='green')
ax2.set_xlabel('K (Number of Recommendations)', fontsize=12)
ax2.set_ylabel('Recall@K', fontsize=12)
ax2.set_title('Recall vs K', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('results/baseline/metrics_plot.png', dpi=300, bbox_inches='tight')
print("✓ Saved: results/baseline/metrics_plot.png")

# Plot 2: Sample recommendations for a user
sample_user = 0
user_predictions = predicted_matrix[sample_user]
train_venues = set(np.where(train_matrix[sample_user] == 1)[0])
test_venues = set(np.where(test_matrix[sample_user] == 1)[0])

# Mask already visited venues
user_predictions_masked = user_predictions.copy()
user_predictions_masked[list(train_venues)] = -np.inf

# Get top 10 recommendations
top_10 = np.argsort(user_predictions_masked)[-10:][::-1]

plt.figure(figsize=(10, 6))
colors = ['green' if v in test_venues else 'blue' for v in top_10]
plt.barh(range(10), user_predictions[top_10], color=colors)
plt.yticks(range(10), [f"Venue {v}" for v in top_10])
plt.xlabel('Predicted Score', fontsize=12)
plt.ylabel('Recommended Venues', fontsize=12)
plt.title(f'Top 10 Recommendations for Sample User\n(Green = Actually visited in test set)', 
          fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig('results/baseline/sample_recommendations.png', dpi=300, bbox_inches='tight')
print("✓ Saved: results/baseline/sample_recommendations.png")

plt.close('all')

# ==========================================
# FINAL SUMMARY
# ==========================================
print("\n" + "="*60)
print("✓ CENTRALIZED BASELINE COMPLETE!")
print("="*60)
print(f"\nYour baseline recommendation system achieved:")
print(f"  Precision@5:  {results[5]['precision']*100:.2f}%")
print(f"  Precision@10: {results[10]['precision']*100:.2f}%")
print(f"  Recall@5:     {results[5]['recall']*100:.2f}%")
print(f"  Recall@10:    {results[10]['recall']*100:.2f}%")

print(f"\nNext steps:")
print("  1. Review results in: results/baseline/")
print("  2. Check metrics_plot.png")
print("  3. Ready to build federated version!")
print("="*60)

CENTRALIZED POI RECOMMENDATION SYSTEM

Step 1: Loading preprocessed data...
✓ Loaded 227428 check-ins
✓ Interaction matrix shape: (1083, 38333)
  - Users: 1083
  - Venues: 38333
  - Sparsity: 99.78%

Step 2: Splitting data into train/test...
✓ Train matrix: 73245 interactions
✓ Test matrix: 17779 interactions

Step 3: Training recommendation model...
Using Matrix Factorization (NMF)...
✓ Model trained!
  - User features shape: (1083, 20)
  - Venue features shape: (20, 38333)

Step 4: Making predictions...
✓ Predicted matrix shape: (1083, 38333)

Step 5: Evaluating model performance...

Evaluation Metrics:
----------------------------------------
K =  5
  Precision@5: 0.0500 (5.00%)
  Recall@5:    0.0159 (1.59%)

K = 10
  Precision@10: 0.0408 (4.08%)
  Recall@10:    0.0266 (2.66%)

K = 20
  Precision@20: 0.0296 (2.96%)
  Recall@20:    0.0378 (3.78%)

Step 6: Saving model and results...
✓ Saved to results/baseline/

Step 7: Creating visualizations...
✓ Saved: results/baseline/metrics_plo